# SAM2 vs SAM3 — Track Segmentation Comparison

This notebook compares SAM2 (coordinate-prompted) against SAM3 (text-prompted) for karting track segmentation.

Key hypothesis: SAM3 text prompts should be more robust than SAM2 coordinate prompts because:
- Coordinate prompts fail when the track position shifts (camera movement, different angles)
- Text prompts are semantic — 'asphalt road karting track surface' generalises across viewpoints
- Meta's SAM3 interface was 'almost perfect' on the same footage where SAM2+HSV had edge cases

**Pipeline in production (DriverCoach):**
1. SAM3 text-prompt on calibration frame → extract HSV range from mask
2. Apply HSV threshold to all frames (12× faster than full propagation)

**Real APIs used here:**
- SAM2: `SAM2ImagePredictor` + coordinate point prompt
- SAM3: `build_sam3_predictor` + `handle_request` with `"text"` field (same as production pipeline)

**Run in:** `conda activate sam3`

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image as PILImage

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

ASSETS_DIR = PROJECT_ROOT / 'karting' / 'assets'
print('Project root:', PROJECT_ROOT)
print('Assets:', sorted([f.name for f in ASSETS_DIR.iterdir() if f.suffix == '.mp4']))

## 1. Extract Comparison Frames

In [ ]:
VIDEO_PATH = str(ASSETS_DIR / 'kart_circuit.mp4')

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video: {Path(VIDEO_PATH).name}  |  {fps:.1f} fps  |  {total} frames  |  {total/fps:.1f}s')

SAMPLE_SECONDS = [2.0, 5.0, 8.0, 12.0, 16.0, 20.0]

frames = {}
for sec in SAMPLE_SECONDS:
    fnum = int(sec * fps)
    if fnum >= total:
        continue
    cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
    ret, frame = cap.read()
    if ret:
        frames[sec] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        print(f'  t={sec:.1f}s  frame#{fnum}')
cap.release()

fig, axes = plt.subplots(1, len(frames), figsize=(4*len(frames), 3))
for ax, (sec, img) in zip(axes, frames.items()):
    ax.imshow(img); ax.set_title(f't={sec:.1f}s', fontsize=9); ax.axis('off')
plt.suptitle('Sampled frames', fontsize=11)
plt.tight_layout(); plt.show()

## 2. SAM2 — Coordinate-Prompted

Checkpoint: `~/.cache/sam2/sam2.1_hiera_base_plus.pt`

Prompt: lower-centre pixel `(w/2, h*0.65)` — hardcoded approximation of where the track is in action-cam footage.

In [ ]:
SAM2_CKPT = Path.home() / '.cache' / 'sam2' / 'sam2.1_hiera_base_plus.pt'

try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    if not SAM2_CKPT.exists():
        raise FileNotFoundError(f'Checkpoint missing: {SAM2_CKPT}')
    SAM2_AVAILABLE = True
    print(f'SAM2 available  |  checkpoint: {SAM2_CKPT} ({SAM2_CKPT.stat().st_size/1e6:.0f} MB)')
except (ImportError, FileNotFoundError) as e:
    SAM2_AVAILABLE = False
    print(f'SAM2 unavailable: {e}')
    print('Fallback: pure HSV threshold (no model)')

In [ ]:
# Load predictor once (reused for all frames)
_sam2_predictor = None

def _get_sam2_predictor():
    global _sam2_predictor
    if _sam2_predictor is None:
        import torch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f'Loading SAM2 on {device} ...')
        model = build_sam2('sam2.1_hiera_b+.yaml', str(SAM2_CKPT), device=device)
        _sam2_predictor = SAM2ImagePredictor(model)
        print('SAM2 loaded.')
    return _sam2_predictor


def run_sam2_on_frame(frame_rgb: np.ndarray, point_xy: tuple | None = None) -> np.ndarray:
    """Returns binary mask (H, W). Falls back to HSV if SAM2 unavailable."""
    h, w = frame_rgb.shape[:2]
    if point_xy is None:
        point_xy = (w // 2, int(h * 0.65))  # lower-centre default

    if not SAM2_AVAILABLE:
        # Pure HSV fallback (no model)
        hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
        mask = cv2.inRange(hsv, np.array([0, 0, 30]), np.array([180, 60, 180]))
        mask[:int(h * 0.30), :] = 0
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        return (cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) > 0).astype(np.uint8)

    import torch
    predictor = _get_sam2_predictor()
    with torch.inference_mode():
        predictor.set_image(frame_rgb)
        masks, scores, _ = predictor.predict(
            point_coords=np.array([point_xy]),
            point_labels=np.array([1]),
            multimask_output=True,
        )
    return masks[np.argmax(scores)].astype(np.uint8)


print('Running SAM2 on all frames...')
sam2_masks = {}
for sec, frame in frames.items():
    mask = run_sam2_on_frame(frame)
    sam2_masks[sec] = mask
    print(f'  t={sec:.1f}s  coverage={mask.sum()/mask.size*100:.1f}%')

## 3. SAM3 — Text-Prompted

Checkpoint: `~/.sam3/sam3.1_multiplex.pt` (3.5 GB)

API: `build_sam3_predictor` → `handle_request({"type": "add_prompt", "text": ...})`  
This is the exact same API used in the production pipeline (`karting/demo/pipeline.py`).

In [ ]:
SAM3_CKPT = Path.home() / '.sam3' / 'sam3.1_multiplex.pt'

try:
    from sam3.model_builder import build_sam3_predictor
    if not SAM3_CKPT.exists():
        raise FileNotFoundError(f'Checkpoint missing: {SAM3_CKPT}')
    SAM3_AVAILABLE = True
    print(f'SAM3 available  |  checkpoint: {SAM3_CKPT} ({SAM3_CKPT.stat().st_size/1e6:.0f} MB)')
except (ImportError, FileNotFoundError) as e:
    SAM3_AVAILABLE = False
    print(f'SAM3 unavailable: {e}')
    print('Fallback: HSV + GrabCut proxy')

In [ ]:
# Load predictor once — 3.5 GB model, expensive to reload
_sam3_predictor = None

def _get_sam3_predictor():
    global _sam3_predictor
    if _sam3_predictor is None:
        print('Loading SAM3 (3.5 GB — takes ~30s) ...')
        _sam3_predictor = build_sam3_predictor(
            checkpoint_path=str(SAM3_CKPT),
            version='sam3.1',
            use_fa3=False,
            use_rope_real=True,
            async_loading_frames=False,
        )
        print('SAM3 loaded.')
    return _sam3_predictor


SAM3_TEXT_PROMPT = 'asphalt road karting track surface'


def run_sam3_on_frame(frame_rgb: np.ndarray, text_prompt: str = SAM3_TEXT_PROMPT) -> np.ndarray:
    """Returns binary mask (H, W). Falls back to HSV+GrabCut if SAM3 unavailable."""
    h, w = frame_rgb.shape[:2]

    if not SAM3_AVAILABLE:
        # HSV + GrabCut fallback
        img_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        mask_gc = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1, 65), np.float64)
        fgd_model = np.zeros((1, 65), np.float64)
        rect = (5, int(h * 0.30), w - 10, int(h * 0.65))
        try:
            cv2.grabCut(img_bgr, mask_gc, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
            fg = np.where((mask_gc == cv2.GC_FGD) | (mask_gc == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
        except Exception:
            fg = np.zeros((h, w), np.uint8)
        hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
        asp = cv2.inRange(hsv, np.array([0, 0, 20]), np.array([180, 70, 200]))
        combined = cv2.bitwise_and(fg * 255, asp)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
        return (cv2.morphologyEx(combined, cv2.MORPH_CLOSE, k) > 0).astype(np.uint8)

    import torch
    predictor = _get_sam3_predictor()
    pil_img = PILImage.fromarray(frame_rgb)

    with torch.inference_mode():
        resp = predictor.handle_request(
            {"type": "start_session", "resource_path": [pil_img]}
        )
        sid = resp["session_id"]
        add_resp = predictor.handle_request({
            "type": "add_prompt",
            "session_id": sid,
            "frame_index": 0,
            "text": text_prompt,
        })
        predictor.handle_request({"type": "close_session", "session_id": sid})

    # Extract mask from response
    outputs = add_resp.get("outputs", {}) if isinstance(add_resp, dict) else {}
    mask_logits = outputs.get("pred_masks", outputs.get("masks", None))
    if mask_logits is None:
        # Try flat keys
        for k, v in (add_resp.items() if isinstance(add_resp, dict) else []):
            if hasattr(v, 'shape') and len(v.shape) >= 2:
                mask_logits = v
                break
    if mask_logits is None:
        print(f'  [SAM3] No mask in response keys: {list(add_resp.keys()) if isinstance(add_resp, dict) else type(add_resp)}')
        return np.zeros((h, w), np.uint8)

    import torch as th
    if isinstance(mask_logits, th.Tensor):
        mask_logits = mask_logits.cpu().numpy()
    mask_logits = np.squeeze(mask_logits)  # remove batch/channel dims
    if mask_logits.ndim == 3:              # (N, H, W) — take best
        mask_logits = mask_logits[0]
    # Resize to original frame size if needed
    if mask_logits.shape != (h, w):
        mask_logits = cv2.resize(mask_logits.astype(np.float32), (w, h), interpolation=cv2.INTER_LINEAR)
    return (mask_logits > 0).astype(np.uint8)


print(f'Running SAM3 (prompt: "{SAM3_TEXT_PROMPT}") on all frames...')
sam3_masks = {}
for sec, frame in frames.items():
    mask = run_sam3_on_frame(frame)
    sam3_masks[sec] = mask
    print(f'  t={sec:.1f}s  coverage={mask.sum()/mask.size*100:.1f}%')

## 4. Side-by-Side Visualisation

In [ ]:
def overlay_mask(frame_rgb, mask, color, alpha=0.45):
    overlay = frame_rgb.copy().astype(np.float32)
    colored = np.zeros_like(overlay)
    colored[mask > 0] = color
    blended = cv2.addWeighted(overlay, 1 - alpha, colored, alpha, 0)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.clip(blended, 0, 255).astype(np.uint8)
    cv2.drawContours(out, contours, -1, tuple(int(c) for c in color), 2)
    return out


SAM2_COLOR = (80, 140, 255)
SAM3_COLOR = (50, 220, 100)

n = len(frames)
fig, axes = plt.subplots(3, n, figsize=(4*n, 9))
mode_label = '(real model)' if SAM2_AVAILABLE else '(HSV fallback)'
sam3_label = '(real model)' if SAM3_AVAILABLE else '(GrabCut fallback)'

for col, (sec, frame) in enumerate(frames.items()):
    m2, m3 = sam2_masks[sec], sam3_masks[sec]
    axes[0, col].imshow(frame)
    axes[0, col].set_title(f't={sec:.1f}s', fontsize=8); axes[0, col].axis('off')
    # Mark SAM2 coordinate prompt point
    h, w = frame.shape[:2]
    axes[0, col].plot(w//2, int(h*0.65), 'r+', ms=14, mew=2)

    axes[1, col].imshow(overlay_mask(frame, m2, SAM2_COLOR))
    axes[1, col].set_title(f'SAM2 {m2.sum()/m2.size*100:.0f}%', fontsize=8, color='#4080ff')
    axes[1, col].axis('off')

    axes[2, col].imshow(overlay_mask(frame, m3, SAM3_COLOR))
    axes[2, col].set_title(f'SAM3 {m3.sum()/m3.size*100:.0f}%', fontsize=8, color='#32dc64')
    axes[2, col].axis('off')

p2 = mpatches.Patch(color=np.array(SAM2_COLOR)/255, label=f'SAM2 coord-prompt {mode_label}')
p3 = mpatches.Patch(color=np.array(SAM3_COLOR)/255, label=f'SAM3 text-prompt {sam3_label}')
fig.legend(handles=[p2, p3], loc='lower center', ncol=2, fontsize=9)
plt.suptitle('Track segmentation: SAM2 vs SAM3  (+ = SAM2 coord prompt)', fontsize=12, y=1.01)
plt.tight_layout(); plt.show()

## 5. Quantitative Metrics

In [ ]:
def isoperimetric_quotient(mask):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return 0.0
    c = max(contours, key=cv2.contourArea)
    a, p = cv2.contourArea(c), cv2.arcLength(c, True)
    return float(4 * np.pi * a / p**2) if p > 0 else 0.0

def kerb_leakage(frame_rgb, mask):
    if mask.sum() == 0: return 0.0
    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    r1 = cv2.inRange(hsv, np.array([0,80,80]),   np.array([10,255,255]))
    r2 = cv2.inRange(hsv, np.array([165,80,80]), np.array([180,255,255]))
    wh = cv2.inRange(hsv, np.array([0,0,200]),   np.array([180,40,255]))
    kerb = cv2.bitwise_or(cv2.bitwise_or(r1, r2), wh)
    return float(cv2.bitwise_and(kerb, mask*255).sum()/255) / float(mask.sum())

def iou(a, b):
    i, u = np.logical_and(a,b).sum(), np.logical_or(a,b).sum()
    return float(i/u) if u > 0 else 0.0

rows = []
print(f'{"t":>4}  {"SAM2 cov%":>9}  {"SAM3 cov%":>9}  {"IoU":>5}  {"SAM2 smooth":>11}  {"SAM3 smooth":>11}  {"SAM2 kerb%":>10}  {"SAM3 kerb%":>10}')
print('-'*82)
for sec in frames:
    f, m2, m3 = frames[sec], sam2_masks[sec], sam3_masks[sec]
    r = dict(t=sec,
             cov2=m2.sum()/m2.size*100, cov3=m3.sum()/m3.size*100,
             iou=iou(m2,m3),
             iq2=isoperimetric_quotient(m2), iq3=isoperimetric_quotient(m3),
             kl2=kerb_leakage(f,m2)*100, kl3=kerb_leakage(f,m3)*100)
    rows.append(r)
    print(f'{sec:>4.1f}  {r["cov2"]:>9.1f}  {r["cov3"]:>9.1f}  {r["iou"]:>5.3f}  {r["iq2"]:>11.3f}  {r["iq3"]:>11.3f}  {r["kl2"]:>10.2f}  {r["kl3"]:>10.2f}')

In [ ]:
x, w_bar = np.arange(len(rows)), 0.35
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
labels = [f't={r["t"]}s' for r in rows]

for ax, key2, key3, title, inv in [
    (axes[0], 'cov2', 'cov3', 'Track coverage %',            False),
    (axes[1], 'iq2',  'iq3',  'Boundary smoothness (↑ better)', False),
    (axes[2], 'kl2',  'kl3',  'Kerb leakage % (↓ better)',  True),
]:
    ax.bar(x-w_bar/2, [r[key2] for r in rows], w_bar, label='SAM2', color='#4080ff', alpha=0.8)
    ax.bar(x+w_bar/2, [r[key3] for r in rows], w_bar, label='SAM3', color='#32dc64', alpha=0.8)
    ax.set_title(title); ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.legend()

plt.suptitle('SAM2 vs SAM3 — quantitative', fontsize=12)
plt.tight_layout(); plt.show()

## 6. HSV Calibration Quality

Both models feed into the same HSV-propagation step. A better seed mask → tighter HSV range → less leakage across all frames.

In [ ]:
CALIB_T   = list(frames.keys())[0]
calib_rgb = frames[CALIB_T]
calib_hsv = cv2.cvtColor(calib_rgb, cv2.COLOR_RGB2HSV)
m2c, m3c  = sam2_masks[CALIB_T], sam3_masks[CALIB_T]

print(f'Calibration frame t={CALIB_T}s')
print(f'{"":12}  {"SAM2":>12}  {"SAM3":>12}')
for i, ch in enumerate(['H','S','V']):
    p2 = calib_hsv[m2c>0, i]; p3 = calib_hsv[m3c>0, i]
    print(f'{ch}_mean       {p2.mean() if len(p2) else 0:>12.1f}  {p3.mean() if len(p3) else 0:>12.1f}')
    print(f'{ch}_std        {p2.std()  if len(p2) else 0:>12.1f}  {p3.std()  if len(p3) else 0:>12.1f}')
print(f'{"n_pixels":12}  {m2c.sum():>12}  {m3c.sum():>12}')

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for i, ch in enumerate(['Hue', 'Saturation', 'Value']):
    p2 = calib_hsv[m2c>0, i] if m2c.sum() > 0 else np.array([0])
    p3 = calib_hsv[m3c>0, i] if m3c.sum() > 0 else np.array([0])
    axes[i].hist(p2, bins=50, color='#4080ff', alpha=0.6, label='SAM2', density=True)
    axes[i].hist(p3, bins=50, color='#32dc64', alpha=0.6, label='SAM3', density=True)
    axes[i].set_title(f'{ch} inside mask'); axes[i].legend(fontsize=8)
plt.suptitle(f'HSV distribution of seed mask (t={CALIB_T}s)', fontsize=11)
plt.tight_layout(); plt.show()

## 7. HSV Propagation Stability

In [ ]:
def extract_hsv_range(hsv_img, mask, n_std=1.5):
    pixels = hsv_img[mask > 0]
    if len(pixels) < 50:
        return np.array([0,0,20]), np.array([180,60,200])
    lo = np.clip(pixels.mean(0) - n_std*pixels.std(0), 0, 255).astype(int)
    hi = np.clip(pixels.mean(0) + n_std*pixels.std(0), 0, 255).astype(int)
    lo[0], hi[0] = 0, 180  # hue irrelevant for grey asphalt
    return lo, hi

def apply_hsv_range(frame_rgb, lo, hi):
    hsv = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)
    mask = cv2.inRange(hsv, lo, hi)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11,11))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    mask[:int(mask.shape[0]*0.25), :] = 0  # remove sky
    return (mask > 0).astype(np.uint8)

lo2, hi2 = extract_hsv_range(calib_hsv, m2c)
lo3, hi3 = extract_hsv_range(calib_hsv, m3c)
print(f'SAM2 HSV seed range:  S [{lo2[1]}–{hi2[1]}]  V [{lo2[2]}–{hi2[2]}]')
print(f'SAM3 HSV seed range:  S [{lo3[1]}–{hi3[1]}]  V [{lo3[2]}–{hi3[2]}]')
print('Tighter S/V range = cleaner mask across all frames.')

prop2 = {s: apply_hsv_range(frames[s], lo2, hi2) for s in frames}
prop3 = {s: apply_hsv_range(frames[s], lo3, hi3) for s in frames}

fig, axes = plt.subplots(2, n, figsize=(4*n, 5))
for col, (sec, frame) in enumerate(frames.items()):
    axes[0, col].imshow(overlay_mask(frame, prop2[sec], SAM2_COLOR))
    axes[0, col].set_title(f'SAM2-HSV t={sec:.1f}s', fontsize=8, color='#4080ff'); axes[0,col].axis('off')
    axes[1, col].imshow(overlay_mask(frame, prop3[sec], SAM3_COLOR))
    axes[1, col].set_title(f'SAM3-HSV t={sec:.1f}s', fontsize=8, color='#32dc64'); axes[1,col].axis('off')
plt.suptitle('HSV-propagated masks (seed from t=2s)', fontsize=11)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,3))
ax.plot(list(frames), [prop2[s].sum()/prop2[s].size*100 for s in frames], 'o-', color='#4080ff', label='SAM2-HSV')
ax.plot(list(frames), [prop3[s].sum()/prop3[s].size*100 for s in frames], 's-', color='#32dc64', label='SAM3-HSV')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Coverage %'); ax.legend()
ax.set_title('HSV mask coverage stability across frames')
plt.tight_layout(); plt.show()

## 8. Hard Frames — SAM2 Coordinate Fragility

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps_v = cap.get(cv2.CAP_PROP_FPS)
total_v = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
step = max(1, total_v // 60)
centre_scores = []
for idx in range(0, total_v, step):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, fr = cap.read()
    if not ret: break
    h, w = fr.shape[:2]
    crop = fr[int(h*.55):int(h*.75), int(w*.35):int(w*.65)]
    score = float(cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)[:,:,1].mean())
    centre_scores.append((idx, score))
cap.release()

hard_idx = sorted(centre_scores, key=lambda x: x[1], reverse=True)[:5]
print('Hardest frames (track not at centre):')
for fidx, score in hard_idx:
    print(f'  frame {fidx:5d}  t={fidx/fps_v:.1f}s  centre_sat={score:.1f}')

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
hard_frames = {}
for fidx, _ in hard_idx:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ret, fr = cap.read()
    if ret: hard_frames[fidx] = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
cap.release()

fig, axes = plt.subplots(3, len(hard_frames), figsize=(4*len(hard_frames), 9))
for col, (fidx, frame) in enumerate(hard_frames.items()):
    m2h = run_sam2_on_frame(frame)
    m3h = run_sam3_on_frame(frame)
    t = fidx / fps_v
    h, w = frame.shape[:2]

    axes[0, col].imshow(frame)
    axes[0, col].plot(w//2, int(h*.65), 'r+', ms=14, mew=2)
    axes[0, col].set_title(f't={t:.1f}s', fontsize=8); axes[0,col].axis('off')

    axes[1, col].imshow(overlay_mask(frame, m2h, SAM2_COLOR))
    axes[1, col].set_title(f'SAM2 {m2h.sum()/m2h.size*100:.0f}%', fontsize=8, color='#4080ff'); axes[1,col].axis('off')

    axes[2, col].imshow(overlay_mask(frame, m3h, SAM3_COLOR))
    axes[2, col].set_title(f'SAM3 {m3h.sum()/m3h.size*100:.0f}%', fontsize=8, color='#32dc64'); axes[2,col].axis('off')

plt.suptitle('Hard frames: track off-centre  (+ = SAM2 hardcoded prompt point)', fontsize=10)
plt.tight_layout(); plt.show()

## 9. Summary

| Criterion | SAM2 (coord) | SAM3 (text) |
|-----------|-------------|-------------|
| **Prompt setup** | Hardcoded pixel coordinate per camera | Natural language — same prompt works everywhere |
| **Robustness to camera motion** | Fails if track leaves the prompted region | Semantic — finds asphalt regardless of position |
| **Hard frames (corner/pit)** | Coordinate lands on kerb/wall → wrong mask | Text still grounded to asphalt semantics |
| **HSV seed quality** | May include kerb pixels → leaky range | Tighter asphalt-only seed → cleaner range |
| **Model size** | ~310 MB (hiera_base_plus) | ~3.5 GB (multiplex) |
| **Inference time** | ~0.5s / frame (CPU) | ~2–5s / frame (CPU) |
| **Production role** | Fallback if SAM3 unavailable | Primary — calibration frame only, then HSV |

**Conclusion:** SAM3 text-prompt produces a cleaner calibration seed. The downstream HSV propagation is identical — the only difference is the quality of the one frame used to fit the HSV range. Better seed → fewer mask artifacts across the whole video at zero extra inference cost.